In [ ]:
from collections import deque, defaultdict

In [ ]:
class BufferPool:
    def __init__(self, size, policy="LRU"):
        self.size = size
        self.policy = policy.upper()
        self.frames = []
        self.page_table = {}
        self.time = 0
        self.last_used = {}
        self.history = defaultdict(list)
        self.clock = []
        self.ref_bit = {}
        self.clock_hand = 0
        self.A1in = deque()
        self.Am = deque()

    def fetch_page(self, page_id):
        self.time += 1

        if page_id in self.page_table:
            print(f"{page_id}: HIT")
            self._hit(page_id)
        else:
            print(f"{page_id}: MISS")
            if len(self.frames) >= self.size:
                self.evict()
            self._add_page(page_id)

        self._print_state()

    def _hit(self, page_id):
        self.last_used[page_id] = self.time
        self.history[page_id].append(self.time)

        if self.policy == "LRU":
            self.frames.remove(page_id)
            self.frames.append(page_id)

        elif self.policy == "CLOCK":
            self.ref_bit[page_id] = 1

        elif self.policy == "2Q":
            if page_id in self.A1in:
                self.A1in.remove(page_id)
                self.Am.append(page_id)
            elif page_id in self.Am:
                self.Am.remove(page_id)
                self.Am.append(page_id)

    def _add_page(self, page_id):
        self.frames.append(page_id)
        self.page_table[page_id] = True
        self.last_used[page_id] = self.time
        self.history[page_id].append(self.time)

        if self.policy == "CLOCK":
            self.clock.append(page_id)
            self.ref_bit[page_id] = 1

        elif self.policy == "2Q":
            self.A1in.append(page_id)

    def evict(self):
        if self.policy == "LRU":
            victim = min(self.frames, key=lambda p: self.last_used[p])

        elif self.policy == "LRU-2":
            victim = self._evict_lru2()

        elif self.policy == "CLOCK":
            victim = self._evict_clock()

        elif self.policy == "2Q":
            victim = self._evict_2q()

        else:
            raise ValueError("Invalid policy")

        print(f"Evicting {victim} ({self.policy})")

        self.frames.remove(victim)
        del self.page_table[victim]
        self.last_used.pop(victim, None)
        self.history.pop(victim, None)

        if self.policy == "CLOCK":
            self.clock.remove(victim)
            self.ref_bit.pop(victim, None)

        elif self.policy == "2Q":
            if victim in self.A1in:
                self.A1in.remove(victim)
            elif victim in self.Am:
                self.Am.remove(victim)

    def _evict_lru2(self):
        candidates = []

        for page in self.frames:
            accesses = self.history[page]

            if len(accesses) < 2:
                kth = accesses[0]
                group = 0
            else:
                kth = accesses[-2]
                group = 1

            candidates.append((group, kth, page))

        candidates.sort()
        return candidates[0][2]

    def _evict_clock(self):
        while True:
            page = self.clock[self.clock_hand]

            if self.ref_bit[page] == 0:
                victim = page
                self.clock_hand = (self.clock_hand + 1) % len(self.clock)
                return victim
            else:
                self.ref_bit[page] = 0
                self.clock_hand = (self.clock_hand + 1) % len(self.clock)

    def _evict_2q(self):
        if self.A1in:
            return self.A1in[0]
        else:
            return self.Am[0]

    def _print_state(self):
        print("Buffer:", self.frames)

        if self.policy == "2Q":
            print("A1in:", list(self.A1in))
            print("Am  :", list(self.Am))

        print("-" * 40)

In [ ]:
pages = [1, 2, 3, 1, 4, 2, 5]

print("===== LRU =====")
bp = BufferPool(size=3, policy="LRU")
for p in pages:
    bp.fetch_page(p)

===== LRU =====
1: MISS
Buffer: [1]
----------------------------------------
2: MISS
Buffer: [1, 2]
----------------------------------------
3: MISS
Buffer: [1, 2, 3]
----------------------------------------
1: HIT
Buffer: [2, 3, 1]
----------------------------------------
4: MISS
Evicting 2 (LRU)
Buffer: [3, 1, 4]
----------------------------------------
2: MISS
Evicting 3 (LRU)
Buffer: [1, 4, 2]
----------------------------------------
5: MISS
Evicting 1 (LRU)
Buffer: [4, 2, 5]
----------------------------------------


In [ ]:
pages = [1, 2, 3, 1, 4, 2, 5]

print("===== LRU-k(2) ====")
bp = BufferPool(size=3, policy="LRU-2")
for p in pages:
    bp.fetch_page(p)

===== LRU-k(2) ====
1: MISS
Buffer: [1]
----------------------------------------
2: MISS
Buffer: [1, 2]
----------------------------------------
3: MISS
Buffer: [1, 2, 3]
----------------------------------------
1: HIT
Buffer: [1, 2, 3]
----------------------------------------
4: MISS
Evicting 2 (LRU-2)
Buffer: [1, 3, 4]
----------------------------------------
2: MISS
Evicting 3 (LRU-2)
Buffer: [1, 4, 2]
----------------------------------------
5: MISS
Evicting 4 (LRU-2)
Buffer: [1, 2, 5]
----------------------------------------


In [ ]:
pages = [1, 2, 3, 1, 4, 2, 5]

print("===== CLOCK =====")
bp = BufferPool(size=3, policy="CLOCK")
for p in pages:
    bp.fetch_page(p)

===== CLOCK =====
1: MISS
Buffer: [1]
----------------------------------------
2: MISS
Buffer: [1, 2]
----------------------------------------
3: MISS
Buffer: [1, 2, 3]
----------------------------------------
1: HIT
Buffer: [1, 2, 3]
----------------------------------------
4: MISS
Evicting 1 (CLOCK)
Buffer: [2, 3, 4]
----------------------------------------
2: HIT
Buffer: [2, 3, 4]
----------------------------------------
5: MISS
Evicting 3 (CLOCK)
Buffer: [2, 4, 5]
----------------------------------------


In [ ]:
pages = [1, 2, 3, 1, 4, 2, 5]

print("===== 2Q =====")
bp = BufferPool(size=3, policy="2Q")
for p in pages:
    bp.fetch_page(p)

===== 2Q =====
1: MISS
Buffer: [1]
A1in: [1]
Am  : []
----------------------------------------
2: MISS
Buffer: [1, 2]
A1in: [1, 2]
Am  : []
----------------------------------------
3: MISS
Buffer: [1, 2, 3]
A1in: [1, 2, 3]
Am  : []
----------------------------------------
1: HIT
Buffer: [1, 2, 3]
A1in: [2, 3]
Am  : [1]
----------------------------------------
4: MISS
Evicting 2 (2Q)
Buffer: [1, 3, 4]
A1in: [3, 4]
Am  : [1]
----------------------------------------
2: MISS
Evicting 3 (2Q)
Buffer: [1, 4, 2]
A1in: [4, 2]
Am  : [1]
----------------------------------------
5: MISS
Evicting 4 (2Q)
Buffer: [1, 2, 5]
A1in: [2, 5]
Am  : [1]
----------------------------------------
